# Smart MCQ Solver Challenge – Milestone 1

## Exploratory Data Analysis and TF-IDF Baseline




# Introduction

Multiple Choice Question Answering is a common NLP task where a model must identify the most relevant answer among several options.

### Objectives

- Understand the dataset structure
- Perform basic text preprocessing
- Explore answer distributions
- Build a TF-IDF based similarity system
- Evaluate answer ranking using MAP@3


## Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import string

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    ENGLISH_STOP_WORDS
)

from sklearn.metrics.pairwise import cosine_similarity


## Load Dataset

In [2]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
# Local testing:
# train = pd.read_csv('train.csv')

print('Dataset Shape:', train.shape)
train.head()


Dataset Shape: (2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## Dataset Overview

In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


# Question 1

### Frequency Distribution of Correct Answers

This helps us understand whether the dataset is balanced across answer choices.


In [4]:
answer_counts = train['answer'].value_counts()

print(answer_counts)

most_freq = answer_counts.max()
least_freq = answer_counts.min()

print('\nMost Frequent Count:', most_freq)
print('Least Frequent Count:', least_freq)
print('Answer:', most_freq + least_freq)


answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Most Frequent Count: 490
Least Frequent Count: 324
Answer: 814


# Question 2

### Vocabulary Size After Cleaning

Steps:
1. Convert text to lowercase
2. Remove punctuation
3. Split by whitespace
4. Count unique words


In [5]:
def clean_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

clean_prompts = train['prompt'].fillna('').apply(clean_text)

vocab = set()

for text in clean_prompts:
    vocab.update(text.split())

print('Vocabulary Size:', len(vocab))


Vocabulary Size: 859


# Question 3

### Stop Word Removal

Stop words are common words such as *the, is, are, and* that usually carry little semantic meaning.


In [6]:
row1_prompt = clean_prompts.iloc[0]

filtered_words = [
    word for word in row1_prompt.split()
    if word not in ENGLISH_STOP_WORDS
]

print('Remaining Words:', len(filtered_words))
print(filtered_words)


Remaining Words: 13
['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']


# TF-IDF Vectorization

TF-IDF (Term Frequency–Inverse Document Frequency) converts text into numerical vectors.

Words that appear frequently in a document but rarely across the corpus receive higher importance.


# Question 4

### TF-IDF Vocabulary Size


In [7]:
combined_text = (
    train['prompt'].fillna('') + ' ' +
    train['A'].fillna('') + ' ' +
    train['B'].fillna('') + ' ' +
    train['C'].fillna('') + ' ' +
    train['D'].fillna('') + ' ' +
    train['E'].fillna('')
)

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text)

print('TF-IDF Feature Count:', len(vectorizer.get_feature_names_out()))


TF-IDF Feature Count: 2762


# Cosine Similarity

Cosine similarity measures how similar two vectors are.

- 1 → identical
- 0 → unrelated


# Question 5

### Similarity Between Prompt and Option A for Row ID 1


In [8]:
prompt_vec = vectorizer.transform([train.loc[0, 'prompt']])
option_a_vec = vectorizer.transform([train.loc[0, 'A']])

similarity = cosine_similarity(prompt_vec, option_a_vec)[0][0]

print('Cosine Similarity:', round(similarity, 4))


Cosine Similarity: 0.272


# Question 6

### Highest Similarity Prediction Accuracy

For each row, calculate similarity between the prompt and all five options.
The option with the highest similarity is selected as the prediction.


In [9]:
labels = ['A','B','C','D','E']

correct_predictions = 0

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row['prompt']])

    scores = {}

    for label in labels:
        option_vec = vectorizer.transform([row[label]])
        scores[label] = cosine_similarity(prompt_vec, option_vec)[0][0]

    prediction = max(scores, key=scores.get)

    if prediction == row['answer']:
        correct_predictions += 1

accuracy = (correct_predictions / len(train)) * 100

print('Percentage:', accuracy)


Percentage: 13.55


# MAP@3 Evaluation

MAP@3 rewards models that rank the correct answer earlier in the prediction list.


In [10]:
def map3_single(actual, predictions):

    for rank, pred in enumerate(predictions[:3], start=1):
        if pred == actual:
            return 1 / rank

    return 0


# Questions 7 and 8

Manual MAP@3 verification examples.


In [11]:
print('Q7:', map3_single('C', ['C','A','B']))
print('Q8:', map3_single('B', ['D','B','E']))


Q7: 1.0
Q8: 0.5


# Question 9

### Majority Class Baseline


In [12]:
frequency_order = train['answer'].value_counts().index.tolist()

top3_predictions = frequency_order[:3]

scores = []

for actual in train['answer']:
    scores.append(map3_single(actual, top3_predictions))

majority_map3 = np.mean(scores)

print('Top 3 Classes:', top3_predictions)
print('Majority Baseline MAP@3:', majority_map3)


Top 3 Classes: ['B', 'C', 'A']
Majority Baseline MAP@3: 0.42125


# Question 10

### TF-IDF Similarity Pipeline

1. Compute prompt-option similarities
2. Rank options
3. Select top 3 answers
4. Calculate MAP@3


In [13]:
labels = ['A','B','C','D','E']

all_scores = []

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row['prompt']])

    similarities = {}

    for label in labels:
        option_vec = vectorizer.transform([row[label]])
        similarities[label] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

    ranked_predictions = sorted(
        similarities,
        key=similarities.get,
        reverse=True
    )[:3]

    all_scores.append(
        map3_single(row['answer'], ranked_predictions)
    )

tfidf_map3 = np.mean(all_scores)

print('TF-IDF MAP@3:', tfidf_map3)


TF-IDF MAP@3: 0.2961666666666667


# Conclusion

### Summary

- Explored the dataset structure
- Analyzed answer distribution
- Performed text preprocessing
- Calculated vocabulary statistics
- Applied TF-IDF vectorization
- Measured cosine similarity
- Evaluated predictions using MAP@3
- Built Majority Class and TF-IDF baselines

The TF-IDF baseline provides a useful benchmark for comparison with future deep learning and transformer-based models.
